<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c41-header.png" alt="Nextia Learning · Speech AI: Transcribe, Understand and Speak" width="100%">

# Solution: Why VAD makes recognition better

**[Why VAD makes recognition better](https://learning.nextia-ai.com/courses/speech/m03/why-vad-makes-recognition-better/)** · Speech AI: Transcribe, Understand and Speak · Module 3, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** find text that Whisper wrote where nobody spoke, measure what voice activity detection in front of Whisper does to the word error rate and to the compute time, and tune the detector's threshold for the phone line.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells. No account, no key. The live part downloads Whisper tiny (about 75 MB) the first time. |
| **You should know** | Silero VAD and its threshold, from [WebRTC VAD and Silero VAD](https://learning.nextia-ai.com/courses/speech/m03/webrtc-vad-and-silero-vad/); insertions and deletions, from [Word error rate, step by step](https://learning.nextia-ai.com/courses/speech/m02/word-error-rate-step-by-step/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, SciPy 1.18.1, soundfile 0.14.0 and matplotlib 3.11.2 (scikit-learn 1.9.1, jiwer 4.0.0 and faster-whisper 1.2.1 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/speech/m03/why-vad-makes-recognition-better/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M03-L03-why-vad-makes-recognition-better/why-vad-makes-recognition-better-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, scipy 1.18.1, soundfile 0.14.0, faster-whisper 1.2.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time. faster-whisper brings its own engine (CTranslate2) and the Silero voice detector: the first run of this cell can take 2 minutes.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 scipy==1.18.1 soundfile==0.14.0 matplotlib==3.11.2 faster-whisper==1.2.1
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'scipy', 'soundfile', 'faster-whisper']:
    print(name, version(name))

## Setup: get the Larkfield calls

Run the next cell. It downloads the files that this lesson needs (about 7 MB) into a folder `larkfield-calls`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-calls` (in Colab: the **Files** panel on the left), then run the cell again.

Where the calls come from: Larkfield is an invented shop. Our code wrote the scripts, and open text-to-speech voices (Piper and Kokoro) read them. Every call is **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/data/dataset-card.md) names each voice and its licence. The calls are free to reuse under CC BY 4.0.

In [ ]:
# Setup: download the files into larkfield-calls/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/"
FILES = {
    "data/calls/calls.jsonl": "22fe4e147bf62b4ff6c3b6b4a1cc559b048e12a0c2b3c8098eb72982f2d9e695",
    "data/recordings/asr/whisper-small.jsonl": "953f41bdba51fe75c411960af3757a8c545b29670c87ac9f04b25f824ff3cf6d",
    "data/recordings/asr/whisper-small-vad.jsonl": "eef95186c3a51321675dacf9b78faa081db1c408b9e338125ad0966d2798d6c7",
    "data/calls/phone/C-0251.wav": "d6c47e1efc3fbff3041f918b6cc33ce32a940829d6d6407147afdd4ef73c8a17",
    "data/calls/phone/C-0204.wav": "2fc187d2960de73f32beb554c90781e0a2853520dfce12582d47de0e5f6911e5",
    "data/calls/phone/C-0022.wav": "f1958f7b460c5f040dfd0f84a31cb83677ee66b25b514df21a086d9d16db5a51",
    "data/calls/phone/C-0044.wav": "2c9e0b36c65b1e013cd7f583768ad55ae077e13469a41060d93fcfc0e2f86b84",
    "data/calls/phone/C-0181.wav": "6bce67f941673d66ec1a2cf271c4c7872721a7936dbb841ac927bc856ab96726",
    "data/calls/phone/C-0085.wav": "cd71e22d4f97b8d90de8a940b6de2e8b7dcc10639a3f77fc76f8da85482c1dd7",
    "data/calls/phone/C-0296.wav": "cd11258073aafd3ce7b5788a850f7d057de97f508162ea049b3ff23e10945a31",
    "data/calls/phone/C-0191.wav": "d91a5a1b132cd844a670e7b8fb602613df77f49cd9d23de098141f4024e98d39",
    "compare.py": "db394eb044e0423706644ba7c8e6451e4d21a268293b868032918ea2f30db85b",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-calls").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-calls and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-calls.")

## Setup: helpers

The next cell loads NumPy as `np`, soundfile as `sf`, SciPy's `signal`, matplotlib, the audio player `Audio`, and the course helper as `cp` (normalisation, word error rate and the comparison table). It defines a few small helpers:

- `load_calls()` reads the labels of every call; `reference(call)` joins its turns into the reference transcript.
- `read_audio(path, sr, channel)` reads a recording as numbers between -1 and 1; `dbfs(x)` gives its loudness.
- `true_turns(call)` gives the true (start, end, speaker) of each turn, from the first word to the last word.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: calls, audio, and checks that do not show the answer.
import hashlib, json, warnings
from math import gcd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display
import compare as cp
warnings.filterwarnings("ignore")

def load_calls(path="data/calls/calls.jsonl"):
    """The labels of every call, by call ID."""
    rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
    return {c["call_id"]: c for c in rows}

def reference(call, speaker=None):
    """The reference transcript: the text of all turns (or of one speaker's turns), in time order."""
    return " ".join(t["text"] for t in call["turns"] if speaker in (None, t["speaker"]))

def read_audio(path, sr=None, channel=None):
    """(numbers between -1 and 1, samples a second). channel=None mixes the channels; 0 or 1 keeps one side."""
    x, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    x = x.mean(axis=1) if channel is None else x[:, channel]
    if sr and sr != file_sr:
        g = gcd(file_sr, sr)
        x = signal.resample_poly(x, sr // g, file_sr // g).astype(np.float32)
    return x, sr or file_sr

def dbfs(x):
    """Loudness in decibels below full scale: 0 is the loudest; speech on a phone is often -30 to -15."""
    return float(20 * np.log10(np.sqrt(np.mean(np.square(x))) + 1e-9))

def true_turns(call):
    """The true turns: (start, end, speaker), from the first word to the last word of each turn."""
    return [(t["words"][0]["start"], t["words"][-1]["end"], t["speaker"]) if t.get("words")
            else (t["start"], t["end"], t["speaker"]) for t in call["turns"]]

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Words that nobody said

In call C-0251, the caller says "Hold on, let me find it." Then the caller is silent for 21.4 seconds while looking for the order number. Only quiet background talk is on the line: other synthetic voices in the caller's room. The next cell loads the recorded transcripts of Whisper small on the phone calls, without VAD and with faster-whisper's VAD, and prints what Whisper small wrote in that wait.

In [ ]:
calls = load_calls()
test = [c for c in calls.values() if c["split"] == "test"]
def recorded(method):
    rows = [json.loads(line) for line in open(f"data/recordings/asr/{method}.jsonl", encoding="utf-8") if line.strip()]
    return {r["call_id"]: r for r in rows if r["kind"] == "phone"}
no_vad, with_vad = recorded("whisper-small"), recorded("whisper-small-vad")
text = no_vad["C-0251"]["text"]
start = text.index("Hold on, let me find it.") + len("Hold on, let me find it.")
end = text.index("Sorry about that")
print("Whisper small wrote, inside the caller's 21-second wait:")
print(text[start:end].strip())

> **Check.** You should see several sentences that nobody in the call said to Larkfield, for example "Could you please tell me how to train you to buy a pair of shoes?". Whisper turned a quiet murmur into confident sentences.

## 2. VAD in front of Whisper, on the 60 test calls

`whisper-small-vad` is the same model with `vad_filter=True`: faster-whisper runs Silero VAD first, keeps only the speech regions, and transcribes those.

> **Predict.** With VAD, will the word error rate go down or up?

In [ ]:
def score(rows):
    s = cp.corpus_scores([(reference(c), rows[c["call_id"]]["text"], c["language"]) for c in test])
    rtf = sum(rows[c["call_id"]]["seconds"] for c in test) / sum(c["duration"] for c in test)
    return {"wer": round(s["wer"], 3), "substitutions": s["sub"], "deletions": s["del"], "insertions": s["ins"],
            "rtf": round(rtf, 3)}
print("without VAD:", score(no_vad))
print("with VAD:   ", score(with_vad))

> **Check.** Without VAD: WER 0.086, deletions 114, insertions 146. With VAD: WER 0.141, deletions 510, insertions 148. The WER **goes up**. Insertions hardly change on these calls, but deletions grow from 114 to 510: the VAD threw away speech. The real-time factor (`rtf`) barely moves (0.075 against 0.073), because most of each call is speech.

## 3. Where the lost words are

In [ ]:
lost = []
for c in test:
    pair = lambda rows: cp.corpus_scores([(reference(c), rows[c["call_id"]]["text"], c["language"])])
    lost.append((pair(with_vad)["del"] - pair(no_vad)["del"], c["call_id"], c["language"], c["noise"]))
lost.sort(reverse=True)
for n, call_id, language, noise in lost[:3]:
    print(f"{call_id} ({language}, {noise}): {n} more words deleted with VAD")
print("calls that lost words:", sum(1 for n, *_ in lost if n > 0), "of", len(test))

> **Check.** The call that lost the most is C-0194 (63 words). 31 of the 60 calls lost words. The lost words are quiet phone speech that Silero, at its default threshold of 0.5, did not call speech.

## 4. What the VAD keeps of call C-0251

> **Predict.** The caller is silent for 21.4 seconds, but the background talk goes on. How much of the call will the VAD keep?

In [ ]:
from faster_whisper.vad import VadOptions, get_speech_timestamps
call = calls["C-0251"]
x16, _ = read_audio("data/calls/phone/C-0251.wav", sr=16000)
for threshold in (0.5, 0.3):
    opts = VadOptions(threshold=threshold)   # faster-whisper's defaults: 2 s of silence ends a region, 0.4 s padding
    spans = [(t["start"] / 16000, t["end"] / 16000) for t in get_speech_timestamps(x16, opts, sampling_rate=16000)]
    kept = sum(e - s for s, e in spans)
    print(f"threshold {threshold}: {len(spans)} regions, {kept:.1f} of {call['duration']:.1f} s kept")

> **Check.** At 0.5, the VAD keeps 56.7 of the 65.828 seconds in 3 regions; at 0.3, 65.7 seconds in 1 regions. It keeps most of the wait, because background talk **is** speech. Yet in the recording with VAD, Whisper small wrote nothing in the wait. faster-whisper also cuts the audio into different pieces when the VAD is on, and invented text is fragile: small changes make it appear or disappear. Measure it on many calls.

## 5. Live: Whisper tiny with and without VAD

Now run a recogniser yourself. The next cell downloads Whisper tiny (about 75 MB, once) and transcribes eight validation calls without VAD and with the VAD at 0.5. It takes about a minute.

In [ ]:
from faster_whisper import WhisperModel
from huggingface_hub import snapshot_download
import time
model = WhisperModel(snapshot_download("Systran/faster-whisper-tiny", revision="d90ca5fe260221311c53c58e660288d3deb8d356"),
                     device="cpu", compute_type="int8", cpu_threads=4)
live_calls = ['C-0251', 'C-0204', 'C-0022', 'C-0044', 'C-0181', 'C-0085', 'C-0296', 'C-0191']
audio16 = {c: read_audio(f"data/calls/phone/{c}.wav", sr=16000)[0] for c in live_calls}

def run_tiny(threshold=None):
    """Whisper tiny on the live calls: no VAD (None) or faster-whisper's VAD at a threshold. WER and RTF."""
    pairs, seconds = [], 0.0
    for call_id in live_calls:
        c = calls[call_id]
        t0 = time.perf_counter()
        segments, _ = model.transcribe(audio16[call_id], language=c["language"], beam_size=1,
                                       vad_filter=threshold is not None,
                                       vad_parameters=None if threshold is None else VadOptions(threshold=threshold),
                                       condition_on_previous_text=False)
        hyp = " ".join(s.text.strip() for s in segments)
        seconds += time.perf_counter() - t0
        pairs.append((reference(c), hyp, c["language"]))
    s = cp.corpus_scores(pairs)
    return {"wer": round(s["wer"], 3), "deletions": s["del"], "insertions": s["ins"],
            "rtf": round(seconds / sum(calls[c]["duration"] for c in live_calls), 3)}

run_tiny()   # the first run warms the model up: do not count it
print("no VAD:       ", run_tiny())
print("VAD at 0.5:   ", run_tiny(0.5))

> **Check.** Without VAD, WER 0.094 (deletions 15). With the VAD at 0.5, WER 0.232 (deletions 160). Your times (`rtf`) will differ; the words should be the same.

## 6. Your turn: tune the threshold

> **Your turn.** Run `run_tiny` with each threshold in `[0.5, 0.4, 0.3, 0.2]`. Keep each WER in a dictionary `wers` (threshold → WER). Put the threshold with the lowest WER in `best` (if two thresholds tie, take the higher one: it cuts more).

In [ ]:
thresholds = [0.5, 0.4, 0.3, 0.2]
wers = {}
for threshold in thresholds:
    r = run_tiny(threshold)
    wers[threshold] = r["wer"]
    print(threshold, r)
best = min(thresholds, key=lambda t: (wers[t], -t))
print("best:", best)

In [ ]:
expect('best', best, 0.2)

In [ ]:
expect("its WER", None if best is ... else wers[best], 0.092, tolerance=0.002)

> **Check.** The WERs are {0.5: 0.232, 0.4: 0.149, 0.3: 0.113, 0.2: 0.092}, so the best threshold here is 0.2. A lower threshold gives the recogniser back the quiet speech. These are eight calls and the smallest Whisper, so the lesson page checks the same idea with Whisper small on all 50 validation calls, and then once on the test calls.

## Next

You found invented text, saw the VAD raise the word error rate with its default threshold, and tuned the threshold. The [lesson page](https://learning.nextia-ai.com/courses/speech/m03/why-vad-makes-recognition-better/) has the measurements with Whisper small, the cost per audio hour and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [From sound to text](https://learning.nextia-ai.com/courses/speech/m04/from-sound-to-text/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/speech/m03/why-vad-makes-recognition-better/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every call is synthetic: open text-to-speech voices read scripts that we invented. No real customer and no real person's voice is in this data.